# Notebook 04 — Evaluating Text2Cypher

"It worked on my three questions" is not an evaluation. Here we measure systems on a **benchmark** of 22 questions with **gold Cypher** written and checked by hand (`benchmark.json`), and compare:

| System | Prompt | Entity hints | Examples | Retry |
|---|---|---|---|---|
| `schema-only` | schema | – | – | – |
| `zero-shot` | schema + domain notes | – | – | – |
| `+linking` | schema + notes | ✅ | – | ✅ |
| `full` | schema + notes | ✅ | ✅ (k=4) | ✅ |

**Metrics**
* **validity** – the query passes the guard and runs without error;
* **exact** – the predicted rows equal the gold rows (column names and order ignored);
* **covers** – same number of rows and each gold row is contained in a predicted row (allows extra columns such as a name next to an ID);
* **value recall** – share of gold values found anywhere in the prediction (partial credit).

In [ ]:
import os, sys, json, re, time, math
from pathlib import Path

# --- Connection settings (override with environment variables if you prefer) ---
NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")     # <- change me
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE") or None          # None = default database

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL       = os.getenv("T2C_MODEL", "qwen2.5-coder:7b")      # any chat model you pulled with `ollama pull`
EMBED_MODEL = os.getenv("T2C_EMBED_MODEL", "nomic-embed-text")

print("Neo4j :", NEO4J_URI, "| Ollama:", OLLAMA_HOST, "| model:", MODEL)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 120)

from t2c import Neo4jGraph, extract_schema, schema_to_text

graph = Neo4jGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
schema_text = schema_to_text(extract_schema(graph))
print("Connected. Schema description:", len(schema_text), "characters")

from t2c import (OllamaLLM, build_system_prompt, EntityLinker, ExampleSelector, Text2Cypher,
                 ZeroShot, load_json)

llm = OllamaLLM(MODEL, host=OLLAMA_HOST, temperature=0.0, num_ctx=8192)

### Step 1 — The benchmark
**What it does:** loads the 22 test questions and shows them with their difficulty tag (`easy`, `medium`, `hard`, `hard-alias`, `hard-rule`, `hard-temporal`, `hard-ambiguity`).
**Teaching point:** a good benchmark covers the *failure modes* we identified in notebook 02, not just random questions.

In [ ]:
benchmark = load_json("benchmark.json")
bench_df = pd.DataFrame(benchmark)
display(bench_df[["id", "difficulty", "question"]])
bench_df["difficulty"].value_counts()

### Step 2 — Gold answers come from the graph, not from a file
**What it does:** executes every gold query and shows how many rows it returns.
**Why:** if the data changes, the expected answers change automatically. We store gold **queries**, never hard-coded answer lists.

In [ ]:
for item in benchmark[:5]:
    print(item["id"], item["question"], "->", graph.run_read(item["cypher"]))

### Step 3 — Why not compare the query text?
**What it does:** runs two *different* queries that are both correct for "Which candidates claim ML?".
**Point:** string comparison of Cypher would mark one of them wrong. We compare **results** instead (*execution accuracy*), which is what the user cares about. Its weakness: a wrong query can return the right rows by luck on a small graph — mention this when presenting the scores.

In [ ]:
q1 = "MATCH (:Alias {key:'ml'})-[:ALIAS_OF]->(s)<-[:ABOUT_SKILL]-()<-[:MAKES_CLAIM]-(c) RETURN c.candidateId AS id"
q2 = """MATCH (c:Candidate)-[:MAKES_CLAIM]->(cl:SkillClaim)-[:ABOUT_SKILL]->(s:Skill {name:'Machine Learning'})
RETURN c.name AS who, c.candidateId AS candidate"""
print(graph.run_read(q1)); print(graph.run_read(q2))

### Step 4 — The comparison functions
**What it does:**
* `normalize_value` makes values comparable across drivers (case, numbers as floats, dates/datetimes as text);
* `row_signature` turns a row into the **set of its values** (so `{id:'C01'}` and `{candidate:'C01', who:'Amina Idrissi'}` can be compared);
* `compare_results` computes *exact*, *covers* and *value recall*.

**Discuss:** these choices are a policy. Set-based rows ignore duplicates and ordering; is that OK for "in chronological order" questions? (B15 — no! An exercise below.)

In [ ]:
def _flatten(value):
    if isinstance(value, (list, tuple, set)):
        for v in value:
            yield from _flatten(v)
    elif isinstance(value, dict):
        for v in value.values():
            yield from _flatten(v)
    else:
        yield value

def normalize_value(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return str(value).lower()
    if isinstance(value, (int, float)):
        return repr(float(value))
    text = str(value).strip().lower()
    stamp = re.match(r"(\d{4}-\d{2}-\d{2})[t ](\d{2}:\d{2}:\d{2})", text)   # datetimes from any driver
    return f"{stamp.group(1)} {stamp.group(2)}" if stamp else text

def row_signature(row: dict) -> frozenset:
    """A row becomes the set of its (flattened, normalised) values: column names and order are ignored."""
    return frozenset(normalize_value(v) for v in _flatten(list(row.values())) if v is not None)

def compare_results(gold: list[dict], pred: list[dict] | None) -> dict:
    """Three views of execution accuracy:
    exact      : same set of rows (value sets), regardless of column names/order
    covers     : same number of rows and every gold row is contained in some predicted row
                 (the prediction may return extra columns, e.g. a name next to an ID)
    value_recall: share of all gold values that appear somewhere in the prediction."""
    if pred is None:
        return {"exact": False, "covers": False, "value_recall": 0.0}
    g = {row_signature(r) for r in gold}
    p = {row_signature(r) for r in pred}
    exact = g == p
    covers = len(g) == len(p) and all(any(gr <= pr for pr in p) for gr in g)
    gold_values = set().union(*g) if g else set()
    pred_values = set().union(*p) if p else set()
    recall = len(gold_values & pred_values) / len(gold_values) if gold_values else float(not pred_values)
    return {"exact": exact, "covers": exact or covers, "value_recall": round(recall, 3)}

print(compare_results([{"a": "C01"}], [{"x": "C01", "name": "Amina"}]))      # covers, not exact
print(compare_results([{"a": "C01"}], [{"a": "C01"}, {"a": "C02"}]))          # extra row -> fails
print(compare_results([{"a": "C01"}, {"a": "C02"}], [{"ids": ["C01", "C02"]}]))  # collected list: recall only

### Step 5 — The benchmark runner
**What it does:** for each question: compute the gold rows, call the system, compare, and record validity, scores, attempts, latency and the generated query. The result is a pandas DataFrame — one row per question.

In [ ]:
def run_benchmark(system, benchmark: list[dict], graph, label: str = "") -> "pd.DataFrame":
    """system(question) must return a dict like Text2Cypher.__call__ does."""
    import pandas as pd
    records = []
    for item in benchmark:
        gold_rows = graph.run_read(item["cypher"])
        out = system(item["question"])
        scores = compare_results(gold_rows, out["rows"])
        records.append({"system": label, "id": item["id"], "difficulty": item.get("difficulty", ""),
                        "question": item["question"], "valid": out["error"] is None,
                        **scores, "attempts": out["attempts"], "seconds": out["seconds"],
                        "pred_cypher": out["cypher"], "error": out["error"]})
    return pd.DataFrame(records)

### Step 6 — Build the four systems
**What it does:** creates the systems of the table above with the same model. `ZeroShot` has the same interface as `Text2Cypher` (it returns a dict with `rows`, `error`, …) but makes a single attempt.

In [ ]:
full_prompt = build_system_prompt(schema_text)
schema_only_prompt = build_system_prompt(schema_text, notes="")
linker = EntityLinker(graph)
examples = load_json("examples.json")
try:
    selector = ExampleSelector(examples, embed_fn=lambda t: llm.embed(t, model=EMBED_MODEL))
except Exception as exc:
    print("No embeddings, lexical selection:", exc)
    selector = ExampleSelector(examples)

systems = {
    "schema-only": ZeroShot(graph, llm, schema_only_prompt),
    "zero-shot":   ZeroShot(graph, llm, full_prompt),
    "+linking":    Text2Cypher(graph, llm, full_prompt, linker=linker, selector=None, max_retries=2),
    "full":        Text2Cypher(graph, llm, full_prompt, linker=linker, selector=selector, k=4, max_retries=2),
}

### Step 7 — Run the evaluation
**What it does:** runs 4 systems × 22 questions. On a laptop with a 7B model expect roughly **5–15 minutes** (about 2–10 s per call, more with retries).
Tip for class: run it before the lesson and load `results_<model>.csv` instead.

In [ ]:
results = pd.concat([run_benchmark(system, benchmark, graph, label=name) for name, system in systems.items()],
                    ignore_index=True)
results["model"] = MODEL
results.to_csv(f"results_{MODEL.replace(':', '_').replace('/', '_')}.csv", index=False)
results.head()

### Step 8 — Summary table and chart
**What it does:** averages every metric per system (booleans become percentages) and plots them.
**How to read it:** the gap between *validity* and *exact* is the share of queries that run but answer the wrong question — the dangerous errors.

In [ ]:
order = list(systems)
summary = (results.groupby("system")[["valid", "exact", "covers", "value_recall", "attempts", "seconds"]]
                  .mean().reindex(order).round(3))
display(summary)

ax = summary[["valid", "exact", "covers", "value_recall"]].plot.bar(figsize=(9, 4), rot=0)
ax.set_ylim(0, 1.05); ax.set_ylabel("score"); ax.set_title(f"Text2Cypher on TalentBridge — {MODEL}")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

### Step 9 — Where does each layer help?
**What it does:** pivots *covers* accuracy by difficulty × system. Typically linking helps `hard-alias`/`hard-ambiguity`, and examples help `hard-rule`.

In [ ]:
pivot = results.pivot_table(index="difficulty", columns="system", values="covers", aggfunc="mean")[order]
pivot.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1)

### Step 10 — Error analysis
**What it does:** for the `full` system, lists every question it got wrong with the predicted query, the gold query and both results.
**Teaching activity:** have students classify each failure with the taxonomy of notebook 02 and propose the cheapest fix (rule, example, linker entry, or model change).

In [ ]:
gold_by_id = {b["id"]: b for b in benchmark}
failures = results[(results.system == "full") & (~results.covers)]
print(f"{len(failures)} failures out of {len(benchmark)}")
for _, r in failures.iterrows():
    g = gold_by_id[r.id]
    print(f"\n=== {r.id} [{r.difficulty}] {r.question}")
    print("PRED:", r.pred_cypher)
    print("GOLD:", g["cypher"])
    if r.error:
        print("ERROR:", r.error[:300])
    else:
        print("pred rows:", graph.run_read(r.pred_cypher)[:5])
    print("gold rows:", graph.run_read(g["cypher"])[:5])

### Step 11 (optional) — Compare models
**What it does:** runs the `full` system with every model listed in `MODELS` that is installed in Ollama, then compares accuracy and latency. Pull models first, e.g. `ollama pull llama3.1:8b`.
**Discuss:** accuracy vs. speed vs. memory. A bigger model with no examples vs. a small model with good examples — which wins on *your* laptop?

In [ ]:
from t2c import list_ollama_models
MODELS = ["qwen2.5-coder:7b", "llama3.1:8b", "mistral:7b"]
installed = set(list_ollama_models(OLLAMA_HOST))

per_model = []
for m in [m for m in MODELS if m in installed]:
    m_llm = OllamaLLM(m, host=OLLAMA_HOST, temperature=0.0, num_ctx=8192)
    sys_m = Text2Cypher(graph, m_llm, full_prompt, linker=linker, selector=selector, k=4)
    df = run_benchmark(sys_m, benchmark, graph, label="full"); df["model"] = m
    per_model.append(df)

if per_model:
    cmp = pd.concat(per_model).groupby("model")[["valid", "exact", "covers", "value_recall", "seconds"]].mean().round(3)
    display(cmp)
else:
    print("None of", MODELS, "is installed.")

## Discussion: limits of this evaluation
* **22 questions is small.** One question = 4.5 points. Never claim "system A is better" from a 1–2 question difference; rerun with other models and report the spread.
* **Execution accuracy can be fooled** on a small graph (right rows for the wrong reason). Read the queries, not only the scores.
* **Order and duplicates are ignored** by our set-based comparison.
* **Template leakage:** some examples share patterns with test questions. Try removing example X06 and re-measuring B12.
* **LLM-as-judge** (asking a second model whether the answer is correct) is an alternative when gold answers are expensive — but it brings its own biases.

## Exercises
1. Make the comparison order-aware for questions whose text contains "chronological" or "in order".
2. Add 5 questions to `benchmark.json` (with gold Cypher you have checked!) that target a weakness found in Step 10.
3. Measure the effect of `k` (0, 2, 4, 8 examples) on accuracy and latency; plot it.